# Notebook 32: Weinberg Angle, CKM Phase, and Coupling Constants

**Paper IV, Sections 3--5, 9--10.** Verifies:

1. Central charge c = 12 b(N) for N=3..15
2. Weinberg angle: sin^2 theta_W = 3/11 = 0.2727
3. CKM phase: delta = 70.2 deg = (1/2) log cosh(pi)
4. |V_cb| = 0.044 (BF barrier transmission)
5. Strong coupling: alpha_s = 1/(4 pi sqrt(2)) = 0.0563
6. Coupling lock: alpha/(8 pi G) = 1/(2 pi^2)

In [ ]:
import sys
sys.path.insert(0, '../src')

import math
from math import pi, sqrt, sin, cos, log, exp, cosh, tanh

from planetary_polygons.extensions.standard_model_gauge import (
    b_exact, central_charge, weinberg_angle_cs_threshold, casimir
)
from planetary_polygons.extensions.ckm_mixing import (
    eta_invariant_phase, bf_barrier_transmission
)

assertion_count = 0
def check(condition, msg):
    global assertion_count
    assert condition, f'FAILED: {msg}'
    assertion_count += 1
    print(f'  [ok] {msg}')

## 1. Central Charge: c = 12 b(N)

b(N) = N(N+1)/12 - ln 2 + ln(N)/(N-1)

c = 12 b(N) is the kinetic coefficient of the breathing mode.

In [ ]:
print(f'{"N":>4} {"b(N)":>12} {"c(N) = 12b":>14}')
print('-' * 32)
for N in range(3, 16):
    b = b_exact(N)
    c = central_charge(N)
    print(f'{N:4d} {b:12.4f} {c:14.4f}')

# Check specific values at N=7 and N=11
c7 = central_charge(7)
c11 = central_charge(11)
print(f'\nKey values:')
print(f'  c(7)  = {c7:.4f}')
print(f'  c(11) = {c11:.4f}')

# Verify the formula: b(N) = N(N+1)/12 - ln2 + ln(N)/(N-1)
for N in [7, 11]:
    b_manual = N*(N+1)/12 - log(2) + log(N)/(N-1)
    check(abs(b_manual - b_exact(N)) < 1e-12, f'b({N}) formula matches implementation')

check(abs(c7 - 51.57) < 0.01, 'c(7) approx 51.57')
check(abs(c11 - 126.56) < 0.1, 'c(11) approx 126.56')

## 2. Weinberg Angle: sin^2 theta_W = 3/11

From conformal dimensions at the orbifold fixed point:

- h_Y = Q^2/K = (1/2)^2/1 = 1/4  (U(1), h_dual = 0)
- h_W = j(j+1)/(k + h_dual) = 2/(1+2) = 2/3  (SU(2), h_dual = 2)
- sin^2 theta_W = h_Y / (h_Y + h_W) = (1/4) / (1/4 + 2/3) = (1/4) / (11/12) = 3/11

In [ ]:
result = weinberg_angle_cs_threshold()

print('=== Weinberg angle from CS threshold couplings ===')
print()

# Step by step
Q = 0.5   # U(1)_K charge = m*/N = 2/4
K = 1     # U(1) level
j = 1     # SU(2) spin (from f(2,4) = 2 = j(j+1))
k_su2 = 1  # SU(2) CS level
h_su2 = 2  # dual Coxeter number of SU(2)

print(f'Step 1: Q = m*/N = 2/4 = {Q}')
print(f'Step 2: K = {K} (U(1) level, free boson)')
print(f'Step 3: j = {j} (from f(2,4) = {casimir(2,4)} = j(j+1), j=1)')
print(f'Step 4: k + h_dual(SU(2)) = {k_su2} + {h_su2} = {k_su2 + h_su2}')

# Conformal weights
h_Y = Q**2 / K   # = 1/4
h_W = j*(j+1) / (k_su2 + h_su2)  # = 2/3

print(f'\nConformal weights:')
print(f'  h_Y = Q^2/K = ({Q})^2/{K} = {h_Y}')
print(f'  h_W = j(j+1)/(k+h_dual) = {j}*{j+1}/{k_su2+h_su2} = {h_W:.6f}')

check(abs(h_Y - 0.25) < 1e-10, 'h_Y = 1/4')
check(abs(h_W - 2/3) < 1e-10, 'h_W = 2/3')

# Weinberg angle
sin2_theta = h_Y / (h_Y + h_W)
print(f'\nsin^2 theta_W = h_Y / (h_Y + h_W)')
print(f'             = {h_Y} / ({h_Y} + {h_W:.6f})')
print(f'             = {h_Y} / {h_Y + h_W:.6f}')
print(f'             = (1/4) / (11/12)')
print(f'             = 3/11')
print(f'             = {sin2_theta:.6f}')
print(f'             = {3/11:.6f}  (exact fraction)')

check(abs(sin2_theta - 3/11) < 1e-10, 'sin^2 theta_W = 3/11 exactly')
check(abs(3/11 - 0.2727) < 0.001, '3/11 approx 0.2727')

# Compare to API result
check(abs(result['sin2_theta_W'] - 3/11) < 1e-10, 'API result matches 3/11')

## 3. CKM Phase: delta = (1/2) log cosh(pi) = 70.2 deg

The CKM phase is determined by the integrated scattering phase on H^2:

delta = integral_0^1 Im psi(1/2 + it) dt = (1/2) log cosh(pi * 1) = 70.2 deg

where the digamma identity Im psi(1/2 + im) = (pi/2) tanh(pi m) gives
the antiderivative (1/2) log cosh(pi t).

In [ ]:
print('=== CKM phase from integrated scattering phase ===')
print()

# The digamma identity: Im psi(1/2 + im) = (pi/2) tanh(pi m)
print('Digamma identity: Im psi(1/2 + it) = (pi/2) tanh(pi t)')
print()

# Antiderivative of (pi/2) tanh(pi t) is (1/2) log cosh(pi t)
# Check: d/dt [(1/2) log cosh(pi t)] = (1/2) * pi * sinh(pi t)/cosh(pi t)
#       = (pi/2) tanh(pi t)  ----  correct!
print('Antiderivative: integral Im psi(1/2 + it) dt = (1/2) log cosh(pi t)')
print()
print('Verification: d/dt [(1/2) log cosh(pi t)]')
print('            = (1/2) * pi * sinh(pi t)/cosh(pi t)')
print('            = (pi/2) tanh(pi t)  [matches the integrand]')

# Numerical check of derivative
t_test = 0.7
eps = 1e-7
numerical_deriv = (0.5*log(cosh(pi*(t_test+eps))) - 0.5*log(cosh(pi*(t_test-eps)))) / (2*eps)
exact_deriv = (pi/2) * tanh(pi * t_test)
check(abs(numerical_deriv - exact_deriv) < 1e-5,
      f'd/dt [(1/2) log cosh(pi t)] = (pi/2) tanh(pi t) at t={t_test}')

# The integral from 0 to Delta_m = 1 (the T3 split: mu4_down - mu4_up = 3/2 - 1/2 = 1)
Delta_m = 1.0  # = mu4_down - mu4_up = 1.5 - 0.5 = 1
print(f'\nBF crossing range: Delta_m = mu4_down - mu4_up = 3/2 - 1/2 = {Delta_m}')

# delta = (1/2) log cosh(pi * Delta_m) evaluated at Delta_m = 1
delta_rad = 0.5 * log(cosh(pi * Delta_m))
delta_deg = delta_rad * 180 / pi

print(f'\ndelta = (1/2) log cosh(pi * {Delta_m})')
print(f'      = (1/2) log cosh(pi)')
print(f'      = (1/2) log({cosh(pi):.6f})')
print(f'      = (1/2) x {log(cosh(pi)):.6f}')
print(f'      = {delta_rad:.4f} rad')
print(f'      = {delta_deg:.1f} deg')
print(f'\nObserved: 69 +/- 3 deg')

check(abs(delta_rad - 1.225) < 0.001, 'delta = 1.225 rad')
check(abs(delta_deg - 70.2) < 0.1, 'delta = 70.2 deg')
check(abs(delta_deg - 69) < 3, 'delta within observed range (69 +/- 3 deg)')

In [ ]:
# Cross-check with the API
phase = eta_invariant_phase(N=7)
print(f'API result: delta = {phase["delta_deg"]:.1f} deg = {phase["delta_rad"]:.4f} rad')
check(abs(phase['delta_deg'] - 70.2) < 0.1, 'API delta = 70.2 deg')

# Show the BF-crossing mode dominance
print(f'\nBF-crossing mode dominance: {phase["bf_fraction"]*100:.1f}% of total eta variation')

## 4. |V_cb| = 0.044

From the orbifold image barrier transmission on H^2/Z_7.
The BF-crossing mode (c = 3/2) contributes via the Legendre Q function.

In [ ]:
print('=== |V_cb| from BF barrier transmission ===')
print()

bf = bf_barrier_transmission(N=7)

print(f'Orbifold: H^2 / Z_{bf["N"]}')
print(f'BF-crossing mode: c = {bf["c"]}')
print(f'Legendre order: nu = c - 1/2 = {bf["nu"]}')
print(f'Turning point: rho* = {bf["rho_star"]}')
print(f'Geodesic distance to nearest image: d = {bf["d"]:.4f}')
print(f'cosh(d) = {bf["cosh_d"]:.4f}')
print(f'\nBarrier transmission T_2 = Q_1(cosh d) = {bf["T_2"]:.6f}')
print(f'Self-consistent rho*_sc = {bf["rho_star_sc"]:.4f}')
print(f'Self-consistent T_2_sc = {bf["T_2_sc"]:.6f}')
print(f'\n|V_cb| = sqrt(V_cb_pert * T_2_sc) = sqrt({bf["V_cb_pert"]} * {bf["T_2_sc"]:.6f})')
print(f'       = {bf["V_cb"]:.4f}')
print(f'Observed: 0.0422 +/- 0.0008')

check(abs(bf['V_cb'] - 0.044) < 0.005, '|V_cb| approx 0.044')

## 5. Strong Coupling: alpha_s = 1/(4 pi sqrt(2)) = 0.0563

The derivation chain:
- 3D CS coupling: alpha_s^{3D} = 1/(k + h_dual) = 1/4
- Effective volume: V_eff = k x 4 pi(g-1) x Z(S^3) = 1 x 4 pi x sqrt(2)
- 4D coupling: alpha_s = alpha_s^{3D} x (k+h_dual) / V_eff
  = [1/(k+h_dual)] x (k+h_dual) / V_eff = 1/V_eff
  = 1/(4 pi sqrt(2))

The (k + h_dual) cancels!

In [ ]:
print('=== Strong coupling constant ===')
print()

k = 1        # CS level
h_dual = 3   # dual Coxeter number of SU(3)
g_genus = 2  # genus of Bolza surface
Z_S3_val = sqrt(2)

print(f'3D CS coupling: alpha_s^{{3D}} = 1/(k + h_dual) = 1/({k} + {h_dual}) = {1/(k+h_dual)}')

# Three factors of V_eff
factor_1 = k                          # bare CS level
factor_2 = 4 * pi * (g_genus - 1)    # Bolza surface area
factor_3 = Z_S3_val                   # CS partition function Z(S^3)

V_eff = factor_1 * factor_2 * factor_3
print(f'\nEffective volume V_eff = k x 4pi(g-1) x Z(S^3)')
print(f'  Factor (i):   k = {factor_1}')
print(f'  Factor (ii):  4pi(g-1) = 4pi({g_genus}-1) = 4pi = {factor_2:.4f}')
print(f'  Factor (iii): Z(S^3) = sqrt(2) = {factor_3:.4f}')
print(f'  V_eff = {factor_1} x {factor_2:.4f} x {factor_3:.4f} = {V_eff:.4f}')

# The (k+h^v) cancellation
alpha_s_3D = 1.0 / (k + h_dual)
alpha_s_4D = alpha_s_3D * (k + h_dual) / V_eff

print(f'\nalpha_s^{{4D}} = alpha_s^{{3D}} x (k+h_dual) / V_eff')
print(f'            = [1/(k+h_dual)] x (k+h_dual) / V_eff')
print(f'            = 1 / V_eff')
print(f'            = 1 / (k x 4pi(g-1) x Z(S^3))')
print(f'            = 1 / ({k} x 4pi x sqrt(2))')
print(f'            = 1 / (4pi sqrt(2))')
print(f'            = {alpha_s_4D:.4f}')

alpha_s_exact = 1.0 / (4 * pi * sqrt(2))
print(f'\n1/(4pi sqrt(2)) = {alpha_s_exact:.4f}')

check(abs(alpha_s_4D - alpha_s_exact) < 1e-10, 'alpha_s = 1/(4 pi sqrt(2)) (k+h_dual cancels)')
check(abs(alpha_s_exact - 0.0563) < 0.001, 'alpha_s(M_poly) = 0.0563')

print(f'\nComparison to observed alpha_s(M_Z) = 0.1180 +/- 0.0009:')
print(f'  One-loop running from M_poly to M_Z: alpha_s(M_Z) = 0.115')
print(f'  Two-loop running: alpha_s(M_Z) = 0.122')
print(f'  Observed 0.1180 is bracketed by [0.115, 0.122]')

## 6. Coupling Lock: alpha / (8 pi G) = 1/(2 pi^2)

The gauge coupling alpha = 6/(pi c) and gravitational coupling G = 3/(2c)
give a ratio that is independent of c (and hence of N).

In [ ]:
print('=== Coupling lock: alpha/(8 pi G) = 1/(2 pi^2) ===')
print()

# Algebraic proof
print('Algebraic proof:')
print('  alpha = 6 / (pi * c)')
print('  G     = 3 / (2 * c)')
print('  alpha / (8 pi G) = [6/(pi c)] / [8 pi * 3/(2c)]')
print('                   = [6/(pi c)] / [12 pi / c]')
print('                   = 6 / (12 pi^2)')
print('                   = 1 / (2 pi^2)')
print('  The c cancels: the ratio is N-independent.')

target = 1.0 / (2 * pi**2)
print(f'\n1/(2 pi^2) = {target:.10f}')

# Numerical verification at multiple N values, including c(7) and c(11)
print(f'\n{"N":>4} {"c":>12} {"alpha":>14} {"G":>14} {"alpha/(8piG)":>16} {"1/(2pi^2)":>14}')
print('-' * 76)
for N in range(3, 16):
    c = central_charge(N)
    alpha = 6.0 / (pi * c)
    G = 3.0 / (2 * c)
    ratio = alpha / (8 * pi * G)
    print(f'{N:4d} {c:12.4f} {alpha:14.8f} {G:14.8f} {ratio:16.10f} {target:14.10f}')

# Verify at c=51.57 (N=7) and c=126.56 (N=11)
for N, c_approx in [(7, 51.57), (11, 126.56)]:
    c = central_charge(N)
    alpha = 6.0 / (pi * c)
    G = 3.0 / (2 * c)
    ratio = alpha / (8 * pi * G)
    check(abs(ratio - target) < 1e-10,
          f'alpha/(8piG) = 1/(2pi^2) at N={N} (c={c:.2f})')

# Also verify that the cancellation works for ANY c
for c_test in [1.0, 10.0, 100.0, 1000.0, 0.01]:
    alpha_test = 6.0 / (pi * c_test)
    G_test = 3.0 / (2 * c_test)
    ratio_test = alpha_test / (8 * pi * G_test)
    check(abs(ratio_test - target) < 1e-10,
          f'alpha/(8piG) = 1/(2pi^2) at c={c_test}')

## Summary

In [ ]:
print(f'All {assertion_count} assertions passed.')